In [3]:
!pip install gensim pandas numpy

import os
import pandas as pd
import numpy as np
from gensim.models import Word2Vec, FastText


os.chdir('/content')
!rm -rf NLP_Course
REPO_URL = "https://github.com/dmytroslav/NLP_Course.git"
!git clone $REPO_URL

# Завантажуємо датасет
data_path = '/content/NLP_Course/data/processed_v2.csv'
print(f"Завантаження даних з {data_path}...")
df = pd.read_csv(data_path)
display(df.head(2))

text_column = 'text'

df = df.dropna(subset=[text_column])
df[text_column] = df[text_column].astype(str)

df['tokens'] = df[text_column].apply(lambda x: x.split())

df = df[df['tokens'].map(len) >= 3]

sentences = df['tokens'].tolist()

num_docs = len(sentences)
num_tokens = sum(len(doc) for doc in sentences)

print("\n" + "="*40)
print(f"Використане поле: {text_column}")
print(f"Документів у тренувальному корпусі: {num_docs}")
print(f"Загальна кількість токенів: {num_tokens}")
print("="*40 + "\n")

print("Приклад токенізації (перший документ):")
print(sentences[0][:15], "...\n")

Cloning into 'NLP_Course'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (131/131), done.
remote: Total 186 (delta 79), reused 144 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 2.09 MiB | 9.29 MiB/s, done.
Resolving deltas: 100% (79/79), done.
Завантаження даних з /content/NLP_Course/data/processed_v2.csv...


,text,label
0,"Просто слухайте цей діалог. Ні, це не нарізка ...",True
1,️ Рубль став найнестабільнішою валютою у всьом...,True



Використане поле: text
Документів у тренувальному корпусі: 10734
Загальна кількість токенів: 148068

Приклад токенізації (перший документ):
['Просто', 'слухайте', 'цей', 'діалог.', 'Ні,', 'це', 'не', 'нарізка', 'і', 'фільм', 'Тарантіно.'] ...



In [4]:
import gensim
from gensim.models import Word2Vec, FastText

print("Доочищення токенів (видалення пунктуації та нижній регістр)...")
df['clean_tokens'] = df['text'].apply(lambda x: gensim.utils.simple_preprocess(x, deacc=False))

df = df[df['clean_tokens'].map(len) >= 3]
sentences = df['clean_tokens'].tolist()

print(f"Приклад чистої токенізації: {sentences[0][:10]}...\n")

VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 3
SG = 1

print("Тренування Word2Vec...")
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=SG,
    workers=4
)
print("Word2Vec натреновано!\n")

print("Тренування FastText...")
ft_model = FastText(
    sentences=sentences,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    sg=SG,
    workers=4
)
print("FastText натреновано!")

Доочищення токенів (видалення пунктуації та нижній регістр)...
Приклад чистої токенізації: ['просто', 'слухайте', 'цей', 'діалог', 'ні', 'це', 'не', 'нарізка', 'фільм', 'тарантіно']...

Тренування Word2Vec...
Word2Vec натреновано!

Тренування FastText...
FastText натреновано!


In [5]:
def compare_neighbors(word, topn=5):
    print(f"\n{'='*40}")
    print(f" Аналіз слова: '{word}'")
    print(f"{'='*40}")

    # Word2Vec
    print(f" Word2Vec:")
    try:
        w2v_res = w2v_model.wv.most_similar(word, topn=topn)
        for w, score in w2v_res:
            print(f"   {w} ({score:.3f})")
    except KeyError:
        print(f" Відсутнє у словнику (OOV - Out Of Vocabulary)")

    # FastText
    print(f"\n FastText:")
    try:
        ft_res = ft_model.wv.most_similar(word, topn=topn)
        for w, score in ft_res:
            print(f"   {w} ({score:.3f})")
    except KeyError:
        print(f"   Відсутнє у словнику (OOV)")

words_to_test = [
    "україна",     # 1. Загальне часте
    "ракетний",    # 2. Доменне (часте)
    "тривога",     # 3. Доменне
    "зсу",         # 4. Абревіатура / специфічне
    "ппо",         # 5. Абревіатура
    "блекаут",     # 6. Неологізм / запозичення
    "русня",       # 7. Сленг / noisy text
    "бахмут",      # 8. Власна назва (географія)
    "допомога",    # 9. Загальне слово (політика/економіка)
    "повідомляють" # 10. Шаблонна лексика (з ЛР8)
]

for w in words_to_test:
    compare_neighbors(w)


 Аналіз слова: 'україна'
 Word2Vec:
   якщо (0.961)
   має (0.956)
   росія (0.941)
   але (0.933)
   хоче (0.928)

 FastText:
   україну (0.993)
   україною (0.992)
   українок (0.984)
   українцям (0.976)
   українець (0.976)

 Аналіз слова: 'ракетний'
 Word2Vec:
   мосту (0.986)
   будинку (0.985)
   удар (0.984)
   вінниці (0.982)
   житловому (0.980)

 FastText:
   ракетну (0.996)
   ракета (0.996)
   ракетним (0.995)
   ракетними (0.995)
   ракетно (0.994)

 Аналіз слова: 'тривога'
 Word2Vec:
   оголошена (0.983)
   повітряна (0.983)
   всій (0.964)
   оголошено (0.952)
   повітряну (0.942)

 FastText:
   тривогу (0.996)
   оголошено (0.991)
   тривоги (0.986)
   вінницькій (0.984)
   одеса (0.983)

 Аналіз слова: 'зсу'
 Word2Vec:
   окупантів (0.950)
   знищили (0.945)
   звільнили (0.937)
   обстрілів (0.922)
   загинули (0.922)

 FastText:
   донецька (0.989)
   донецьк (0.986)
   лисичанська (0.985)
   окупантами (0.985)
   сумська (0.984)

 Аналіз слова: 'ппо'
 Word2Vec:
  

| Word | Type | Word2Vec neighbors | FastText neighbors | Useful? | Comment |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **ракетний** | domain | удар, будинку, мосту | ракета, ракетну | **useful** | W2V дав ідеальний семантичний контекст (наслідки удару), FT дав усі словоформи. |
| **повідомляють** | template | пишуть, поширюють, джерела | повідомили, пишуть | **useful** | W2V знайшов ідеальні функціональні синоніми для новинного стилю. |
| **бахмут** | domain/rare | атакує, полоні | бахмуті, яр, дтп | **partly** | FT знайшов Часів Яр, але додав багато буквеного шуму. W2V дав загальний фон. |
| **блекаут** | rare/oov | *[OOV]* | патрушев, учора | **weak** | W2V відкинув слово, FT згенерував абсурдних сусідів через брак даних. |
| **русня** | slang/noisy | полоні, корабля | липня, червня, грудня | **weak** | FT згрупував слова суто за закінченням "-ня" (місяці), проігнорувавши зміст. |